In [ ]:
# A1 · Part 2 — Build a minimal retrieval system.  Starter notebook.
# PE6201 · Emerging AI Technologies. Run top to bottom, then replace the placeholder domain.

# A1 · Part 2 — Build a minimal retrieval system  *(starter)*

**What Part 2 asks of you.** Assemble a small collection of documents from **your** domain —
10 to 30 is enough. Build a system that retrieves relevant passages and uses them to answer
questions. Show it answering **five questions it could not possibly answer without your
documents**. Then **break it**: find a question where retrieval fails or returns the wrong
passage, and explain *why* it failed.

> **The breaking is not optional and it is not a penalty.** Finding the edge of your own system
> is the single most valuable thing you will do in this assignment. A notebook where everything
> works is a notebook that was not pushed hard enough.

**What this notebook is.** A working retrieval-augmented generation (RAG) pipeline on a
**placeholder domain** (houseplant care notes). Everything you must change is marked
**`### REPLACE ME`**.

**Cost.** About 10 short generation calls. Fractions of a cent. Embeddings run locally and free.

**Deliverable from this notebook:** this notebook (run, output visible), five worked examples,
one documented failure, and roughly 400 words.

> **If a cell does not work.** This notebook was written before you ran it, against library
> versions, model names and provider behaviour that change without notice — so treat it as a
> working draft, not gospel. **Read the error, then try to fix it yourself:** debugging is part of
> the work, not a detour from it. Then **report it** — in class or by email — with the cell, the
> full error text, and what you tried, so the shared copy gets fixed and nobody behind you loses
> the same hour. Reporting a problem costs you nothing. The one thing not to do is quietly work
> around a broken cell: if your fix changes what the notebook measures, say so in your writeup.

### Setup

In [1]:
%pip -q install openai scikit-learn sentence-transformers

### Connect to OpenRouter
The **generation** step (writing the answer) goes to OpenRouter with your key. The
**embedding** step (turning text into vectors) runs **locally and free** — OpenRouter serves chat
models, not embedding models, which is exactly the "which layer do you rent?" question from
Class 2.

In [2]:
import os, getpass, time
import numpy as np
from openai import OpenAI

def _get_openrouter_key():
    k = os.environ.get('MY_PRIVATE_OPENROUTER_KEY') or os.environ.get('OPENROUTER_API_KEY')
    if k: return k
    try:
        from google.colab import userdata
        k = userdata.get('OPENROUTER_API_KEY')
        if k: return k
    except Exception:
        pass
    return getpass.getpass('Paste your OpenRouter API key (hidden): ')

API_KEY = _get_openrouter_key()
client  = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=API_KEY)
GEN_MODEL = 'openai/gpt-4o-mini'      # TRY THIS (knob): any model your key can reach
TOKENS = {'in': 0, 'out': 0}

def generate(prompt, system=None, max_new_tokens=300, temperature=0.0):
    msgs = ([{'role': 'system', 'content': system}] if system else []) + \
           [{'role': 'user', 'content': prompt}]
    r = client.chat.completions.create(model=GEN_MODEL, messages=msgs,
                                       temperature=temperature, max_tokens=max_new_tokens)
    if getattr(r, 'usage', None):
        TOKENS['in']  += r.usage.prompt_tokens
        TOKENS['out'] += r.usage.completion_tokens
    return r.choices[0].message.content.strip()

print('connected ·', generate('Reply with exactly the word: ready', max_new_tokens=6))

connected · ready


### Check your remaining credit  —  free, this call is not billed
Your allocation is **fixed and will not be topped up**, so watch it rather than discovering it at
the end. `usage` is what you have spent; `limit` is your cap; the difference is what is left.
Run this at the start of a working session and again at the end.

In [3]:
import urllib.request, json as _json

def credit_check():
    """Ask OpenRouter what is left on your key. Costs nothing."""
    try:
        req = urllib.request.Request('https://openrouter.ai/api/v1/key',
                                     headers={'Authorization': f'Bearer {API_KEY}'})
        d = _json.load(urllib.request.urlopen(req, timeout=20))['data']
        used, cap = d.get('usage'), d.get('limit')
        if cap is None:
            print(f'spent so far: ${used:.4f}   (no cap set on this key)')
        else:
            print(f'spent ${used:.4f} of ${cap:.2f}  ->  ${cap - used:.4f} remaining')
        rl = d.get('rate_limit') or {}
        if rl:
            print(f"rate limit: {rl.get('requests')} requests per {rl.get('interval')}")
    except Exception as e:
        print('could not check credit (not a problem, carry on):', str(e)[:120])

credit_check()

spent $0.0043 of $10.00  ->  $9.9957 remaining
rate limit: -1 requests per 10s


---
## 0 · Five words you need — *start here if you don't code*

| Word | In plain words |
|---|---|
| **Embedding** | A piece of text turned into a list of numbers, arranged so that texts with similar *meaning* end up near each other. |
| **Chunk** | A small slice of a document. We search chunks, not whole documents, so what comes back is the relevant *paragraph* rather than the whole manual. |
| **Retrieval** | Finding the chunks nearest the question. Pure arithmetic — no model involved, no understanding. |
| **Top-k** | How many chunks we fetch. Too few and the answer is missing; too many and the real answer drowns in noise. |
| **Grounding** | Making the model answer **only** from the retrieved text, and say so when the text does not cover it. |

**RAG in one line:** *embed your documents → retrieve the ones nearest the question → let a model
answer from them.* **The model is rented; the retrieval is yours.** That is why retrieval is where
almost every failure lives — and why the failure you document is the interesting part.

---
## 1 · Your documents  ### REPLACE ME

**10 to 30 documents.** Each a short paragraph — a few sentences is fine.

**The one rule that matters:** they must contain things a model *could not already know*. If your
documents are summaries of Wikipedia, the model answers correctly with retrieval switched off, and
you have proved nothing. Good sources: your own notes, internal procedures, club or team records,
a rulebook nobody has digitised, your family's version of anything, recent local information,
specifics with names and numbers in them.

**Test for this later, in Section 4.** We ask the model each question with the documents removed.
If it still gets them right, your document set is too generic — go back and make it specific.

In [31]:
DOCS = ['# Apple iPhone 15 Pro Max — battery and video playback\n'
 'Apple lists iPhone 15 Pro Max video playback as up to 29 hours, and streamed video playback as '
 'up to 25 hours. These are Apple test results and actual results vary with use and settings.\n'
 'Official source: https://support.apple.com/en-sg/111828 (checked 2026-10-02).',
 '# MacBook Pro with M3 Max — unified memory\n'
 "Apple's M3 Max MacBook Pro memory options depend on the chip configuration. Apple lists 96GB for "
 'M3 Max with a 14-core CPU; the M3 Max with a 16-core CPU can be configured up to 128GB. Thus '
 '96GB is a real configuration, but it is not the maximum across M3 Max models.\n'
 'Official sources: https://support.apple.com/en-sg/117736 and '
 'https://www.apple.com/sg/newsroom/2023/10/apple-unveils-new-macbook-pro-featuring-m3-chips/ '
 '(checked 2026-10-02).',
 '# Apple Watch Ultra 2 — water resistance and recreational diving\n'
 'Apple lists water resistance to 100 metres under ISO 22810:2010 and EN13319 recreational scuba '
 'diving to 40 metres (±1 metre accuracy). Apple says not to use it for diving below 40 metres; '
 'compatible third-party dive app and dive safety practices apply. Water resistance can diminish '
 'over time.\n'
 'Official source: https://support.apple.com/en-sg/111832 (checked 2026-10-02).',
 '# Apple Pencil compatibility — M4 iPad Pro\n'
 'Apple Pencil (2nd generation) is not listed as compatible with iPad Pro (M4). Apple lists Apple '
 'Pencil Pro and Apple Pencil (USB-C) as compatible with iPad Pro (M4). Check the exact iPad model '
 'and iPadOS requirements before purchase.\n'
 'Official source: https://support.apple.com/en-sg/108937 (checked 2026-10-02).',
 '# Apple Online Store Singapore — returns\n'
 "Apple's Singapore store says eligible products may be returned within 14 calendar days from "
 'receipt, subject to the terms and product-specific rules. The iPhone/iPad section lists '
 'additional fees and conditions, so do not generalize one return outcome to every product.\n'
 'Official source: https://www.apple.com/sg/shop/help/returns_refund (checked 2026-10-02).',
 '# Apple Trade In Singapore — timing and condition\n'
 "Apple's Singapore Trade In page says its trade-in partner needs to receive the device within 14 "
 'days after the customer gets the trade-in estimate, and the device condition must match the '
 'description. The page does not establish the classroom claims of an 11-day deadline after '
 'receiving a new product, a seven-day revised-quote window, or a two-device online transaction '
 'limit.\n'
 'Official source: https://www.apple.com/sg-edu/shop/trade-in (checked 2026-10-02).',
 '# Apple Store for Education Singapore — annual quantity limits\n'
 "Apple's Singapore Education sales policy lists yearly limits: one desktop, one Mac mini, one "
 'MacBook, one display, two iPads, one Apple Watch with Education Pricing, and two accessories '
 'with Education Pricing. Limits apply across online purchases and Apple Retail Stores.\n'
 'Official source: https://www.apple.com/sg-edu/shop/browse/open/salespolicies/edu (checked '
 '2026-10-02).',
 '# Apple Online Store Singapore — delivery and signature\n'
 "Apple's Singapore Shipping & Delivery page says available options and delivery timeframes are "
 'shown at checkout. Some shipments require a signature; postal letterbox deliveries do not. Apple '
 'does not state a universal S$750 or US$750 threshold in this guidance. Check the specific order '
 'status for its requirement.\n'
 'Official source: https://www.apple.com/sg/shop/help/shipping_delivery (checked 2026-10-02).',
 '# Apple hardware warranty\n'
 'Apple publishes a one-year limited warranty for Apple-branded hardware, subject to the warranty '
 'terms and consumer-law rights. This does not support a blanket promise that every Apple product '
 'has a five-year free replacement warranty. Individual coverage and legal rights depend on '
 'product, location and circumstances.\n'
 'Official source: '
 'https://www.apple.com/sg/legal/warranty/products/warranty-rest-of-apac-simplified-chinese.html '
 '(checked 2026-10-02).',
 '# AppleCare+ — pricing is product and region specific\n'
 'AppleCare+ is an optional service plan with terms that vary by product and region. This corpus '
 'does not include a verified AppleCare+ price for iPhone 15 Pro Max in Singapore. If asked for '
 "that price, say the notes do not provide it and direct the user to Apple's current Singapore "
 'coverage page or checkout.\n'
 'Official overview: https://www.apple.com/sg/legal/sales-support/ (checked 2026-10-02).']


10 documents


### Chunking
Long documents get cut into overlapping slices before embedding. Why: one embedding holds one
*average* meaning, so a long multi-topic document becomes a blurry vector that matches everything
a little and nothing well. Small chunks keep each vector about a single idea. The **overlap** stops
an idea that straddles a boundary from being cut in half.

This is a knob you tune against your evidence, not a setting with a correct value.

In [32]:
# TRY THIS (knob): watch what changes in Section 3 when you move these.
CHUNK_WORDS = 100     # try 15 (tiny) or 300 (whole document = one chunk)
OVERLAP     = 10     # words shared with the next chunk

def chunk(text, size=CHUNK_WORDS, overlap=OVERLAP):
    words, out, i = text.split(), [], 0
    while i < len(words):
        out.append(' '.join(words[i:i+size]))
        if i + size >= len(words):
            break
        i += max(1, size - overlap)
    return out

CHUNKS = [(c, d_i) for d_i, d in enumerate(DOCS) for c in chunk(d)]
print(f'{len(DOCS)} docs -> {len(CHUNKS)} chunks (CHUNK_WORDS={CHUNK_WORDS}, OVERLAP={OVERLAP})')

10 docs -> 22 chunks (CHUNK_WORDS=100, OVERLAP=10)


### Embed and retrieve
The embedder runs locally — a small model downloaded once, no key, no cost. If the download fails
(no internet on the machine, or a locked-down environment), the fallback cell below uses **TF-IDF**
word-overlap instead. TF-IDF works, and it is a useful contrast: it matches *words*, not *meaning*,
so it fails on paraphrase. If you use the fallback, say so in your writeup — it changes which
failures you find.

In [40]:
EMBEDDER = 'sentence-transformers/all-MiniLM-L6-v2'
USING = None
try:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(EMBEDDER)
    def embed(texts):
        v = _m.encode(list(texts), normalize_embeddings=True)
        return np.asarray(v)
    USING = f'local embeddings ({EMBEDDER}) — matches MEANING'
except Exception as e:
    print('embedder unavailable ->', str(e)[:120])
    from sklearn.feature_extraction.text import TfidfVectorizer
    _v = TfidfVectorizer().fit([c for c, _ in CHUNKS])
    def embed(texts):
        m = _v.transform(list(texts)).toarray()
        n = np.linalg.norm(m, axis=1, keepdims=True)
        return m / np.where(n == 0, 1, n)
    USING = 'TF-IDF fallback — matches WORDS, not meaning'

print('using:', USING)
MATRIX = embed([c for c, _ in CHUNKS])

TOP_K = 1            # TRY THIS (knob): 1 (brittle) or 8 (noisy)

def retrieve(question, k=TOP_K):
    q = embed([question])[0]
    scores = MATRIX @ q
    order = np.argsort(-scores)[:k]
    return [(CHUNKS[i][0], CHUNKS[i][1], float(scores[i])) for i in order]

for txt, d_i, s in retrieve('how often is the fiddle leaf fig watered in winter?'):
    print(f'  {s:.3f}  [doc {d_i}]  {txt[:80]}...')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

using: local embeddings (sentence-transformers/all-MiniLM-L6-v2) — matches MEANING
  0.172  [doc 5]  # Standard Return & Refund Policy ## Return Window Customers have **14 calendar ...


### Answer from the retrieved text — and only from it
The instruction to say *"the notes do not say"* is the whole point. Without it the model fills gaps
from its training data and you cannot tell which sentences came from your documents. That is
**silent failure**: a confident, fluent, unsourced answer.

In [41]:
GROUNDED = (
    "Answer using ONLY the notes provided. "
    "If the notes do not contain the answer, reply exactly: The notes do not say. "
    "Do not use any other knowledge. Quote the phrase you relied on."
)

def answer(question, k=TOP_K, show=True):
    hits = retrieve(question, k)
    context = '\n\n'.join(f'[{i+1}] {t}' for i, (t, _, _) in enumerate(hits))
    out = generate(f'NOTES:\n{context}\n\nQUESTION: {question}', system=GROUNDED)
    if show:
        print(f'Q: {question}\nA: {out}\n   (top chunk score {hits[0][2]:.3f}, doc {hits[0][1]})\n')
    return out, hits

answer('When was the fiddle-leaf fig repotted, and into what?')

Q: When was the fiddle-leaf fig repotted, and into what?
A: The notes do not say.
   (top chunk score 0.114, doc 5)



('The notes do not say.',
 [('# Standard Return & Refund Policy ## Return Window Customers have **14 calendar days** from the date of receiving the product to initiate a return for any item purchased directly from the official store. ## Eligibility & Condition Requirements - Items must be returned in their original condition with all original packaging, cables, adapters, and manuals included. - Products showing physical damage, liquid damage, or unauthorized modification are ineligible for return. - Open box items are eligible for a full refund as long as all contents remain intact. ## Refund Processing Times - Once received at the warehouse, inspection takes',
   5,
   0.11425148695707321)])

---
## 2 · Five questions it could not answer without your documents  ### REPLACE ME

Each question should have a **specific** answer that lives in your documents — a date, a name, a
number, a decision and its outcome. "What is a monstera?" is a bad question: the model knows that
already, so a correct answer proves nothing about your retrieval.

In [38]:
### REPLACE ME — five questions answerable ONLY from your documents.
QUESTIONS = [
    "What is the maximum number of trade‑in devices permitted in one online Apple trade‑in transaction?",
    "Within how many calendar days must a customer return their trade‑in device after receiving the new product?",
    "What is the maximum unified memory available for the MacBook Pro M3 Max?",
    "How many hours of video playback can the iPhone 15 Pro Max deliver?",
    "What is the ISO‑standard water‑resistance depth rating of the Apple Watch Ultra 2?",
]
for q in QUESTIONS:
    answer(q)

Q: What is the maximum number of trade‑in devices permitted in one online Apple trade‑in transaction?
A: The maximum number of trade‑in devices permitted in one online Apple trade‑in transaction is "2."
   (top chunk score 0.714, doc 6)

Q: Within how many calendar days must a customer return their trade‑in device after receiving the new product?
A: Customers must mail back the trade‑in device within **11 calendar days** of receiving the new product.
   (top chunk score 0.674, doc 6)

Q: What is the maximum unified memory available for the MacBook Pro M3 Max?
A: The maximum unified memory available for the MacBook Pro M3 Max is **96GB**. "M3 Max supports up to **96GB** Unified Memory and"
   (top chunk score 0.702, doc 2)

Q: How many hours of video playback can the iPhone 15 Pro Max deliver?
A: The notes say: "iPhone 15 Pro Max: Up to **24 hours** of video playback."
   (top chunk score 0.744, doc 0)

Q: What is the ISO‑standard water‑resistance depth rating of the Apple Watch Ultra 2

---
## 3 · Break it  ### REPLACE ME

**This section carries the marks.** You are looking for a question where retrieval brings back the
wrong passage — not where the model writes a clumsy sentence.

Four kinds of break that reliably work. Try all four; keep the one that fails most interestingly.

| Kind | What to try |
|---|---|
| **Vocabulary mismatch** | Ask using a word that never appears in your documents but means the same thing. Devastating for TF-IDF; embeddings usually survive it. |
| **Two documents needed** | Ask something that requires combining facts from two documents. Top-k retrieval fetches the *most similar* chunks, which is not the same as *the set that answers the question*. |
| **Distractor** | Ask about a topic covered in several documents where only one is relevant. Watch the right chunk get outvoted. |
| **Absent but plausible** | Ask something your documents nearly, but do not quite, cover. Does it say "the notes do not say", or does it invent? |

**Diagnose, do not just report.** "It got it wrong" is a Developing answer. "The right chunk ranked
fourth with score 0.31 while an irrelevant chunk scored 0.44 because both contain the word
*water*" is a Proficient one.

In [42]:
### REPLACE ME — your breaking attempts.
BREAKERS = [
    ("vocabulary mismatch", "What maximum amount of onboard RAM can the highest‑spec M3 laptop support?"),
    ("two documents needed", "Can I combine student education pricing with Apple trade‑in online, and what is the trade‑in device limit per single online transaction?"),
    ("distractor", "Which Apple Watch model is certified for recreational scuba diving and what is its general ISO water‑resistance rating?"),
    ("absent but plausible", "How much does Apple charge for the AppleCare+ extended warranty for iPhone 15 Pro Max?"),
]
for kind, q in BREAKERS:
    print(f'--- {kind.upper()} ---')
    out, hits = answer(q)
    print('   retrieved:')
    for t, d_i, s in hits:
        print(f'     {s:.3f} [doc {d_i}] {t[:70]}...')
    print()

--- VOCABULARY MISMATCH ---
Q: What maximum amount of onboard RAM can the highest‑spec M3 laptop support?
A: The notes do not say.
   (top chunk score 0.633, doc 2)

   retrieved:
     0.633 [doc 2] # MacBook Lineup Technical Reference ## Product Models This document c...

--- TWO DOCUMENTS NEEDED ---
Q: Can I combine student education pricing with Apple trade‑in online, and what is the trade‑in device limit per single online transaction?
A: Yes, you can combine student education pricing with an Apple trade-in offer on the same transaction. The notes do not say about the trade-in device limit per single online transaction.
   (top chunk score 0.695, doc 8)

   retrieved:
     0.695 [doc 8] # Promotional Stacking & Higher Education Discount Rules ## Higher Edu...

--- DISTRACTOR ---
Q: Which Apple Watch model is certified for recreational scuba diving and what is its general ISO water‑resistance rating?
A: The notes do not say.
   (top chunk score 0.777, doc 4)

   retrieved:
     0.777

### Diagnose the one you will submit
Fill this in **in prose**, from the scores printed above. This is your documented failure.

In [ ]:
FAILURE_DIAGNOSIS = """
### 
Question that failed:
Can I combine student education pricing with Apple trade‑in online, and what is the trade‑in device limit per single online transaction?

What the system answered:
Yes, you can combine student education pricing with an Apple trade‑in offer on the same transaction. The notes do not say about the trade‑in device limit per single online transaction.

What the correct answer was:
Student education pricing can be combined with Apple trade‑in on the same online purchase, and a maximum of one trade‑in device can be applied per single online transaction.

Which chunk SHOULD have been retrieved, and what did it score:
The chunk containing trade‑in device limit rule from document 6. This required chunk was never retrieved because top‑k is set to 1.

What was retrieved instead, and what did it score:
Top‑1 retrieved chunk (score 0.695 from doc 8). This passage only explains that education pricing can stack with trade‑in, and contains no information regarding how many trade‑in devices are allowed for one online order. Only this single chunk is passed into the LLM context.

WHY (the mechanism — vocabulary, chunk size, top‑k, a distractor, or genuinely absent):
Two‑documents‑needed failure. The question depends on separate facts stored inside two different source documents. Under top‑k=1 setting, retrieval selects only the highest‑similarity chunk. The second necessary fact from another document is excluded entirely from context. The LLM can answer the first half of the question but has no access to the second constraint. Small chunk size further separates facts into more fragments and increases this failure chance.

What you would change to fix it, and what that change would cost you elsewhere:
Increase top‑k value so multiple chunks from different documents can be included in prompt context. The cost is larger prompt size, higher token consumption, longer inference time, and greater risk of injecting irrelevant distracting content into the context window.
"""
print(FAILURE_DIAGNOSIS)



### 
Question that failed:
Can I combine student education pricing with Apple trade‑in online, and what is the trade‑in device limit per single online transaction?

What the system answered:
Yes, you can combine student education pricing with an Apple trade‑in offer on the same transaction. The notes do not say about the trade‑in device limit per single online transaction.

What the correct answer was:
Student education pricing can be combined with Apple trade‑in on the same online purchase, and a maximum of one trade‑in device can be applied per single online transaction.

Which chunk SHOULD have been retrieved, and what did it score:
The chunk containing trade‑in device limit rule from document 6. This required chunk was never retrieved because top‑k is set to 1.

What was retrieved instead, and what did it score:
Top‑1 retrieved chunk (score 0.695 from doc 8). This passage only explains that education pricing can stack with trade‑in, and contains no information regarding how many t

---
## 4 · Two checks worth running before you write

**Check 1 — is retrieval doing anything?** Ask the same five questions with the documents removed.
If the model still answers correctly, your documents are too generic and Part 2 has not been
demonstrated. Fix the documents.

**Check 2 — does the grounding instruction hold?** A system that invents when it should decline is
worse than one that declines too often, because you cannot see it happening.

In [39]:
print('=== CHECK 1: same questions, NO documents ===')
for q in QUESTIONS[:5]:
    print(f'Q: {q}\nA: {generate(q, system="Answer in one short sentence.", max_new_tokens=80)}\n')

print('=== CHECK 2: something your documents definitely do not cover ===')
answer('What is the postal code of the flat?')

=== CHECK 1: same questions, NO documents ===
Q: What is the maximum number of trade‑in devices permitted in one online Apple trade‑in transaction?
A: The maximum number of trade-in devices permitted in one online Apple trade-in transaction is five.

Q: Within how many calendar days must a customer return their trade‑in device after receiving the new product?
A: A customer must return their trade-in device within 14 calendar days after receiving the new product.

Q: What is the maximum unified memory available for the MacBook Pro M3 Max?
A: The maximum unified memory available for the MacBook Pro M3 Max is 128GB.

Q: How many hours of video playback can the iPhone 15 Pro Max deliver?
A: The iPhone 15 Pro Max can deliver up to 29 hours of video playback.

Q: What is the ISO‑standard water‑resistance depth rating of the Apple Watch Ultra 2?
A: The Apple Watch Ultra 2 has an ISO-standard water-resistance depth rating of 100 meters.

=== CHECK 2: something your documents definitely do not 

('The notes do not say.',
 [('# Delivery & Shipping Guidelines ## Shipping Options & Timelines - **Standard Shipping**: Free on all eligible orders. Delivery within **2 to 4 business days** after order processing. - **Courier Express Delivery**: Available in select metropolitan areas for a fixed fee. Delivery within 2 hours or scheduled 2‑hour windows. - **Pickup at Apple Store**: Free option. Available within 2 hours for in‑stock items after order placement. ## Signature Requirements & Address Changes - High‑value orders (exceeding $750 value, including MacBooks, iPhones, and iPads) **require a direct signature** upon delivery. - Address modifications cannot be made once an order status has',
   7,
   0.3069677948951721),
  ('Address modifications cannot be made once an order status has changed to "Preparing for Shipment" or "Shipped". - If delivery fails after 3 attempts, the shipment is returned to the warehouse and automatically processed as a return refund.',
   7,
   0.1865818798

---
## 5 · YOUR TURN

1. Replace `DOCS` with 10–30 documents from your domain that contain things a model cannot know.
2. Replace `QUESTIONS` with five that are answerable only from them. Run and read every answer.
3. Run Check 1. If the model answers without your documents, go back to step 1.
4. Work through all four `BREAKERS`. Keep the most interesting failure and diagnose it properly.
5. Try `CHUNK_WORDS`, `OVERLAP` and `TOP_K` at extremes. Note which failures appear and disappear.

### What your ~400 words must contain

- What your documents are, and why a model could not answer these questions without them.
- The five worked examples (the notebook output is enough; the words are for what they show).
- **The failure, diagnosed** — which chunk should have won, which did, the scores, and the
  mechanism.
- What you would change to fix it, and **what that fix would break** — bigger chunks improve
  context and dilute matching; a larger top-k improves recall and adds noise and tokens. There is
  no free setting, and saying so is the postgraduate move.

In [43]:
print(f'tokens this session: {TOKENS["in"]} in / {TOKENS["out"]} out')

tokens this session: 13642 in / 1095 out
